In [ ]:
!pip install ultralytics supervision opencv-python numpy pandas -q

import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
import supervision as sv
from collections import defaultdict, deque
from google.colab import files
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
import base64

print("✅ Libraries Installed. Ready.")

# ----------------------------
# Zone drawing (UNCHANGED)
# ----------------------------
def drawzone(imagepath, zonename):
    frame = cv2.imread(imagepath)
    h, w = frame.shape[:2]
    scale = 800 / w
    resized = cv2.resize(frame, (int(w * scale), int(h * scale)))

    buf = cv2.imencode(".jpg", resized)[1]
    imgdata = "data:image/jpeg;base64," + base64.b64encode(buf).decode("utf-8")

    js = Javascript("""
    async function draw(imgsrc, zonename) {
      const div = document.createElement('div');
      div.innerHTML = `<h3>Draw ${zonename} (Click corners, Double-click to finish)</h3>
                       <canvas id="c"></canvas><br>
                       <button id="btn">Save & Continue</button>`;
      document.body.appendChild(div);

      const canvas = document.getElementById('c');
      const ctx = canvas.getContext('2d');
      const img = new Image();
      img.src = imgsrc;
      await new Promise(r => img.onload = r);

      canvas.width = img.width;
      canvas.height = img.height;
      ctx.drawImage(img, 0, 0);

      let pts = [];

      canvas.onclick = (e) => {
        const r = canvas.getBoundingClientRect();
        pts.push([Math.round(e.clientX - r.left), Math.round(e.clientY - r.top)]);

        ctx.fillStyle = 'lime';
        ctx.fillRect(pts[pts.length-1][0]-2, pts[pts.length-1][1]-2, 4, 4);

        if (pts.length > 1) {
          ctx.beginPath();
          ctx.moveTo(pts[pts.length-2][0], pts[pts.length-2][1]);
          ctx.lineTo(pts[pts.length-1][0], pts[pts.length-1][1]);
          ctx.strokeStyle = 'lime';
          ctx.stroke();
        }
      };

      canvas.ondblclick = () => {
        if (pts.length > 2) {
          ctx.beginPath();
          ctx.moveTo(pts[pts.length-1][0], pts[pts.length-1][1]);
          ctx.lineTo(pts[0][0], pts[0][1]);
          ctx.strokeStyle = 'lime';
          ctx.stroke();
        }
      };

      await new Promise(r => document.getElementById('btn').onclick = () => {
        div.remove();
        r();
      });

      return pts;
    }
    """)
    display(js)

    pts = eval_js(f"draw('{imgdata}', '{zonename}')")
    return (np.array(pts) / scale).astype(int)

# --- EXECUTE DRAWING ---
VIDEOPATH = "mno.mp4"  # Ensure this matches your file name!
cap = cv2.VideoCapture(VIDEOPATH)
ret, frame = cap.read()

if ret:
    cv2.imwrite("ref.jpg", frame)

    print("Step 1: Draw the FLOOR (Walking Area)")
    FLOORPOLY = drawzone("ref.jpg", "FLOOR ZONE")

    print("Step 2: Draw SHELF A (First Aisle)")
    SHELFAPOLY = drawzone("ref.jpg", "SHELF A")

    print("Step 3: Draw SHELF B (Second Aisle)")
    SHELFBPOLY = drawzone("ref.jpg", "SHELF B")

    print("✅ Zones Saved.")
else:
    print("❌ Error: Could not read video. Upload it first.")

# ----------------------------
# CONFIGURATION (UNCHANGED + only CSV additions + VALIDITEMS expanded)
# ----------------------------
OUTPUTVIDEO = "finalinsightoutput.mp4"
OUTPUTCSV = "customerjourneyreport.csv"

# ✅ ADDED CSV OUTPUTS (new files only; no logic changes)
OUTPUT_TRACKS_CSV = "tracks.csv"
OUTPUT_ZONE_VISITS_CSV = "zone_visits.csv"
OUTPUT_INTERACTIONS_CSV = "interactions.csv"
OUTPUT_FRAME_METRICS_CSV = "frame_metrics.csv"

WRISTCROP = 130
CONSENSUSFRAMES = 1
RETURNFRAMES = 4

# ✅ VALIDITEMS expanded ONLY (still COCO IDs)
# Original had: 24,25,26,28,32,34,35,36,38,39,41,43,44,45 [file:23]
# Added common retail/sports relevant COCO IDs:
# 29=frisbee, 31=snowboard, 33=kite, 37=surfboard, 40=wine glass,
# 42=fork, 46=bowl, 47=banana, 48=apple, 49=sandwich, 50=orange,
# 51=broccoli, 52=carrot, 53=hot dog, 54=pizza, 55=donut, 56=cake,
# 57=chair, 58=couch, 59=potted plant, 60=bed, 61=dining table,
# 62=toilet, 63=tv, 64=laptop, 65=mouse, 66=remote, 67=keyboard,
# 68=cell phone, 69=microwave, 70=oven, 71=toaster, 72=sink,
# 73=refrigerator, 74=book, 75=clock, 76=vase, 77=scissors,
# 78=teddy bear, 79=hair drier, 80=toothbrush
VALIDITEMS = [
    24, 25, 26, 28, 32, 34, 35, 36, 38, 39, 41, 43, 44, 45,
    29, 31, 33, 37, 40, 42, 46, 47, 48, 49, 50, 51, 52, 53,
    54, 55, 56, 57, 58, 59, 60, 61, 63, 64, 65, 66, 67, 68,
    74, 75, 76, 77, 78, 79, 80
]

class Customer:
    def __init__(self, id):
        self.id = id
        self.entrytime = 0
        self.exittime = 0
        self.state = "WALKING"  # States: WALKING, BROWSING, PICKED, RETURNED
        self.currentzone = "FLOOR"
        self.productshandled = []
        self.journey = []
        self.pickbuffer = 0
        self.returnbuffer = 0
        self.history = deque(maxlen=30)

        # ✅ ADDED: data features only (no effect on behavior)
        self.zone_enter_time = {}     # zone_name -> enter_time
        self.zone_total_time = defaultdict(float)  # zone_name -> total dwell seconds
        self.interaction_active = False
        self.interaction_start = None
        self.interaction_zone = None
        self.interaction_item = None

def main():
    # 1. LOAD MODELS (UNCHANGED)
    print("Loading AI Models (ByteTrack Enabled)...")
    posemodel = YOLO("yolov8l-pose.pt")
    objmodel = YOLO("yolov8m.pt")

    cap = cv2.VideoCapture(VIDEOPATH)
    W, H = int(cap.get(3)), int(cap.get(4))
    fps = cap.get(5)

    out = cv2.VideoWriter(
        OUTPUTVIDEO,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (W, H)
    )

    # Zones Setup (UNCHANGED)
    zones = {
        "SHELF_A": sv.PolygonZone(SHELFAPOLY, triggering_anchors=[sv.Position.CENTER]),
        "SHELF_B": sv.PolygonZone(SHELFBPOLY, triggering_anchors=[sv.Position.CENTER]),
    }
    floorpoly = sv.PolygonZone(FLOORPOLY, triggering_anchors=[sv.Position.CENTER])

    customers = {}
    framen = 0
    print(f"Starting Advanced Analysis {W}x{H}...")

    # ✅ ADDED: collectors (only for CSV)
    frame_metrics_rows = []
    zone_visit_rows = []
    interaction_rows = []

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        framen += 1
        currtime = framen / fps

        # --- A. TRACKING (UNCHANGED) ---
        results = posemodel.track(frame, persist=True, tracker="bytetrack.yaml", verbose=False)

        if results[0].boxes.id is not None:
            ids = results[0].boxes.id.int().cpu().numpy()
            boxes = results[0].boxes.xyxy.cpu().numpy()
            kpts = results[0].keypoints.xy.cpu().numpy()

            for i, tid in enumerate(ids):
                # Init Customer (UNCHANGED)
                if tid not in customers:
                    customers[tid] = Customer(tid)
                    customers[tid].entrytime = currtime

                c = customers[tid]
                c.exittime = currtime
                box = boxes[i]

                # Update Movement History (UNCHANGED)
                center = (int((box[0] + box[2]) / 2), int(box[3]))
                c.history.append(center)

                # --- B. ZONE DETECTION (UNCHANGED) ---
                lw, rw = kpts[i][9], kpts[i][10]
                activeshelf = None
                wristpt = None

                for zname, zpoly in [(n, zones[n].polygon) for n in zones]:
                    if cv2.pointPolygonTest(zpoly, (lw[0], lw[1]), False) >= 0:
                        activeshelf, wristpt = zname, lw
                        break
                    elif cv2.pointPolygonTest(zpoly, (rw[0], rw[1]), False) >= 0:
                        activeshelf, wristpt = zname, rw
                        break

                # Update Journey Logic (UNCHANGED)
                currentloc = activeshelf if activeshelf else "FLOOR"
                if c.currentzone != currentloc:
                    # ✅ ADDED: zone visit close/open (no behavior change)
                    prev_zone = c.currentzone
                    if prev_zone in c.zone_enter_time:
                        enter_t = c.zone_enter_time.pop(prev_zone)
                        dwell = max(0.0, currtime - enter_t)
                        c.zone_total_time[prev_zone] += dwell
                        zone_visit_rows.append({
                            "CustomerID": int(c.id),
                            "Zone": prev_zone,
                            "EnterTime": round(enter_t, 3),
                            "ExitTime": round(currtime, 3),
                            "DwellTime": round(dwell, 3),
                        })
                    c.zone_enter_time[currentloc] = currtime

                    c.currentzone = currentloc
                    if currentloc not in c.journey:
                        c.journey.append(currentloc)

                # --- C. BEHAVIOR LOGIC (UNCHANGED) ---
                color = (0, 255, 0)  # Green Walking

                if activeshelf:
                    detecteditem = None

                    if framen % 3 == 0:
                        x, y = int(wristpt[0]), int(wristpt[1])
                        x1, x2 = max(0, x - WRISTCROP), min(W, x + WRISTCROP)
                        y1, y2 = max(0, y - WRISTCROP), min(H, y + WRISTCROP)
                        crop = frame[y1:y2, x1:x2]

                        if crop.size > 0:
                            res = objmodel(crop, verbose=False, conf=0.15)
                            for b in res[0].boxes:
                                clsid = int(b.cls[0])
                                if clsid in VALIDITEMS:
                                    detecteditem = res[0].names[clsid]
                                    cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 255), 1)
                                    break

                    if detecteditem:
                        c.pickbuffer += 1
                        c.returnbuffer = 0
                    else:
                        c.pickbuffer = max(0, c.pickbuffer - 1)
                        if c.state == "PICKED":
                            c.returnbuffer += 1

                    if c.pickbuffer >= CONSENSUSFRAMES:
                        c.state = "PICKED"
                        if detecteditem and detecteditem not in c.productshandled:
                            c.productshandled.append(detecteditem)

                        # ✅ ADDED: open interaction record (no behavior change)
                        if not c.interaction_active:
                            c.interaction_active = True
                            c.interaction_start = currtime
                            c.interaction_zone = activeshelf
                            c.interaction_item = detecteditem

                    elif c.returnbuffer >= RETURNFRAMES and c.state == "PICKED":
                        c.state = "RETURNED"

                        # ✅ ADDED: close interaction as RETURNED
                        if c.interaction_active:
                            interaction_rows.append({
                                "CustomerID": int(c.id),
                                "Zone": c.interaction_zone,
                                "StartTime": round(c.interaction_start, 3) if c.interaction_start else None,
                                "EndTime": round(currtime, 3),
                                "HoldTime": round(currtime - (c.interaction_start or currtime), 3),
                                "Outcome": "RETURNED",
                                "DetectedItem": c.interaction_item,
                            })
                            c.interaction_active = False
                            c.interaction_start = None
                            c.interaction_zone = None
                            c.interaction_item = None

                    else:
                        c.state = "BROWSING"

                    if c.state == "PICKED":
                        color = (0, 0, 255)
                    elif c.state == "RETURNED":
                        color = (255, 255, 0)
                    else:
                        color = (0, 165, 255)

                else:
                    if c.state == "BROWSING":
                        c.state = "WALKING"

                    # ✅ ADDED: if they leave shelf while still PICKED, close as BOUGHT
                    if c.state == "PICKED" and c.interaction_active:
                        interaction_rows.append({
                            "CustomerID": int(c.id),
                            "Zone": c.interaction_zone,
                            "StartTime": round(c.interaction_start, 3) if c.interaction_start else None,
                            "EndTime": round(currtime, 3),
                            "HoldTime": round(currtime - (c.interaction_start or currtime), 3),
                            "Outcome": "BOUGHT",
                            "DetectedItem": c.interaction_item,
                        })
                        c.interaction_active = False
                        c.interaction_start = None
                        c.interaction_zone = None
                        c.interaction_item = None

                    if c.state == "PICKED":
                        color = (0, 0, 255)
                    if c.state == "RETURNED":
                        color = (255, 255, 0)

                # --- D. VISUALIZATION (UNCHANGED) ---
                cv2.rectangle(frame, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), color, 2)

                label = f"ID {tid} {c.state}"
                if c.productshandled:
                    label += f" | {c.productshandled[-1]}"
                cv2.putText(frame, label, (int(box[0]), int(box[1]) - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

                if len(c.history) >= 2:
                    points = np.array(c.history, dtype=np.int32).reshape(-1, 1, 2)
                    cv2.polylines(frame, [points], False, color, 2)

                cv2.polylines(frame, [FLOORPOLY], True, (50, 50, 50), 1)
                cv2.polylines(frame, [SHELFAPOLY], True, (255, 255, 0), 2)
                cv2.polylines(frame, [SHELFBPOLY], True, (255, 255, 0), 2)

                cv2.rectangle(frame, (0, 0), (400, 60), (0, 0, 0), -1)
                cv2.putText(frame, f"Active Customers: {len(ids) if 'ids' in locals() else 0}", (10, 25),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                cv2.putText(frame, "Red=Took  Cyan=Returned  Orange=Browsing", (10, 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)

                # ✅ ADDED: frame metrics row (no behavior change)
                frame_metrics_rows.append({
                    "time_s": round(currtime, 3),
                    "frame_idx": framen,
                    "CustomerID": int(c.id),
                    "state": c.state,
                    "zone": c.currentzone,
                    "center_x": int(center[0]),
                    "center_y": int(center[1]),
                    "lw_x": float(lw[0]),
                    "lw_y": float(lw[1]),
                    "rw_x": float(rw[0]),
                    "rw_y": float(rw[1]),
                })

        out.write(frame)
        if framen % 10 == 0:
            print(f"Processing {currtime:.1f}s...", end="\r")

    cap.release()
    out.release()

    # ✅ ADDED: close open zone visits at end
    final_time = framen / fps if fps else 0.0
    for c in customers.values():
        for z, enter_t in list(c.zone_enter_time.items()):
            dwell = max(0.0, final_time - enter_t)
            c.zone_total_time[z] += dwell
            zone_visit_rows.append({
                "CustomerID": int(c.id),
                "Zone": z,
                "EnterTime": round(enter_t, 3),
                "ExitTime": round(final_time, 3),
                "DwellTime": round(dwell, 3),
            })

    # --- GENERATE SALES INSIGHTS REPORT (UNCHANGED) ---
    rows = []
    for c in customers.values():
        conversion = "Browsed Only"
        if c.state == "PICKED":
            conversion = "Converted Picked"
        elif c.state == "RETURNED":
            conversion = "Interested but Returned"

        journeystr = " - ".join(c.journey) if c.journey else "Floor Only"
        productstr = ", ".join(c.productshandled) if c.productshandled else "None"

        rows.append({
            "CustomerID": int(c.id),
            "TotalDwellTime": round(c.exittime - c.entrytime, 2),
            "FinalStatus": c.state,
            "ConversionType": conversion,
            "ProductsInteracted": productstr,
            "StoreJourneyPath": journeystr,
        })

    df = pd.DataFrame(rows)
    df.to_csv(OUTPUTCSV, index=False)

    # ✅ ADDED: tracks.csv
    tracks_rows = []
    for c in customers.values():
        tracks_rows.append({
            "CustomerID": int(c.id),
            "EntryTime": round(c.entrytime, 3),
            "ExitTime": round(c.exittime, 3),
            "TotalTime": round(c.exittime - c.entrytime, 3),
            "FinalStatus": c.state,
            "ZonesVisited": " - ".join(c.journey) if c.journey else "Floor Only",
            "Products": ", ".join(c.productshandled) if c.productshandled else "None",
        })
    pd.DataFrame(tracks_rows).to_csv(OUTPUT_TRACKS_CSV, index=False)

    # ✅ ADDED: zone_visits.csv
    pd.DataFrame(zone_visit_rows).to_csv(OUTPUT_ZONE_VISITS_CSV, index=False)

    # ✅ ADDED: interactions.csv
    pd.DataFrame(interaction_rows).to_csv(OUTPUT_INTERACTIONS_CSV, index=False)

    # ✅ ADDED: frame_metrics.csv
    pd.DataFrame(frame_metrics_rows).to_csv(OUTPUT_FRAME_METRICS_CSV, index=False)

    print("\n✅ Analysis Complete. Downloading...")
    files.download(OUTPUTVIDEO)
    files.download(OUTPUTCSV)

    # ✅ ADDED downloads
    files.download(OUTPUT_TRACKS_CSV)
    files.download(OUTPUT_ZONE_VISITS_CSV)
    files.download(OUTPUT_INTERACTIONS_CSV)
    files.download(OUTPUT_FRAME_METRICS_CSV)

if __name__ == "__main__":
    if "SHELFAPOLY" in globals():
        main()
    else:
        print("❌ Error: Please run the zone drawing step first!")